In [1]:

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

SEED = 7
rng = np.random.default_rng(SEED)


## Section 0 - Generate synthetic sample data

In [2]:

# -----------------------------
# Universe / metadata generation
# -----------------------------
n_tickers = 180
dates = pd.bdate_range("2018-01-01", "2022-12-30", freq="B")

tickers = [f"STK{i:03d}" for i in range(n_tickers)]
sectors = [
    "Tech", "Health", "Financials", "Industrials", "Consumer",
    "Energy", "Utilities", "Materials", "Comm", "RealEstate"
]
countries = ["US", "CA", "UK"]
exchanges = ["NYSE", "NASDAQ", "TSX", "LSE"]

meta = pd.DataFrame({
    "ticker": tickers,
    "sector": rng.choice(sectors, size=n_tickers, p=np.array([0.17,0.12,0.13,0.11,0.12,0.08,0.08,0.07,0.06,0.06])),
    "country": rng.choice(countries, size=n_tickers, p=[0.78, 0.12, 0.10]),
    "exchange": rng.choice(exchanges, size=n_tickers, p=[0.34, 0.44, 0.12, 0.10]),
    "shares_outstanding_m": rng.uniform(20, 2500, size=n_tickers),
    "base_beta": rng.normal(1.0, 0.22, size=n_tickers),
    "quality_tilt": rng.normal(0, 1, size=n_tickers),
    "value_tilt": rng.normal(0, 1, size=n_tickers),
    "growth_tilt": rng.normal(0, 1, size=n_tickers),
    "liquidity_tilt": rng.normal(0, 1, size=n_tickers),
})
meta["industry"] = meta["sector"] + "_" + rng.integers(1, 5, size=n_tickers).astype(str)

# Add IPO / delist dates to make panel handling more realistic
ipo_idx = rng.integers(0, int(len(dates) * 0.45), size=n_tickers)
active_len = rng.integers(int(len(dates) * 0.45), len(dates), size=n_tickers)
delist_idx = np.minimum(ipo_idx + active_len, len(dates) - 1)

# Keep many names alive through end date
alive_mask = rng.random(n_tickers) < 0.72
delist_idx = np.where(alive_mask, len(dates) - 1, delist_idx)

meta["ipo_date"] = dates[ipo_idx]
meta["delist_date"] = dates[delist_idx]

meta.head()


,ticker,sector,country,exchange,shares_outstanding_m,base_beta,quality_tilt,value_tilt,growth_tilt,liquidity_tilt,industry,ipo_date,delist_date
0,STK000,Consumer,UK,NYSE,"2,309.49677",1.42332,-0.43733,0.74982,0.92671,2.36244,Consumer_4,2019-12-02,2022-12-30
1,STK001,Comm,CA,NASDAQ,"1,647.83174",1.37611,0.30732,-0.63003,1.06253,0.48304,Comm_1,2019-02-15,2022-12-30
2,STK002,Utilities,CA,NYSE,"1,559.84882",1.12451,-0.27792,0.48129,0.34104,-0.71629,Utilities_4,2019-10-14,2022-12-30
3,STK003,Health,US,NASDAQ,"1,923.87399",1.15054,0.12047,1.86832,-2.45766,0.70489,Health_2,2019-07-08,2022-12-30
4,STK004,Financials,UK,NASDAQ,"1,880.64045",0.55400,-0.13209,1.17300,-0.67559,-0.33432,Financials_3,2019-07-23,2022-12-30


In [3]:

# ----------------------------------------
# Simulate market, sector, and stock returns
# ----------------------------------------
market_ret = pd.Series(rng.normal(0.0002, 0.010, len(dates)), index=dates, name="mkt_ret")

sector_shocks = pd.DataFrame(
    rng.normal(0, 0.006, size=(len(dates), len(sectors))),
    index=dates,
    columns=sectors
)

# Mild regime effects
regime = pd.Series(np.where(dates < "2020-03-01", 0,
                    np.where(dates < "2020-09-01", 1,
                    np.where(dates < "2021-10-01", 2, 3))), index=dates)

regime_vol = regime.map({0: 1.0, 1: 1.8, 2: 0.85, 3: 1.1}).astype(float).values
market_ret = market_ret * regime_vol

rows = []
for _, row in meta.iterrows():
    live_mask = (dates >= row["ipo_date"]) & (dates <= row["delist_date"])
    d = dates[live_mask]

    sec = row["sector"]
    n = len(d)
    eps = rng.normal(0, 0.018, n)
    idio = pd.Series(eps, index=d)

    # latent components
    momentum_component = pd.Series(rng.normal(0, 0.0012, n), index=d).rolling(20, min_periods=1).mean()
    value_component = row["value_tilt"] * 0.00012
    quality_component = row["quality_tilt"] * 0.00008
    growth_component = row["growth_tilt"] * 0.00010

    ret = (
        row["base_beta"] * market_ret.loc[d].values
        + sector_shocks.loc[d, sec].values
        + momentum_component.values
        + value_component
        + quality_component
        + growth_component
        + idio.values
    )

    # occasional outliers / event days
    event_days = rng.random(n) < 0.004
    ret[event_days] += rng.normal(0, 0.08, event_days.sum())

    start_px = rng.uniform(8, 180)
    close = start_px * np.exp(np.cumsum(np.log1p(np.clip(ret, -0.95, None))))
    close = np.maximum(close, 0.5)

    volume = (
        rng.lognormal(mean=13.3 + 0.20 * row["liquidity_tilt"], sigma=0.65, size=n)
        * (1 + 15 * np.abs(ret))
    )
    volume = np.maximum(volume, 1000)

    open_px = close / np.exp(np.clip(ret, -0.5, 0.5)) * np.exp(rng.normal(0, 0.0035, n))
    high = np.maximum(open_px, close) * (1 + np.abs(rng.normal(0.004, 0.006, n)))
    low = np.minimum(open_px, close) * np.maximum(0.80, 1 - np.abs(rng.normal(0.004, 0.006, n)))

    rows.append(pd.DataFrame({
        "date": d,
        "ticker": row["ticker"],
        "open": open_px,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
        "ret_1d": ret,
    }))

prices = pd.concat(rows, ignore_index=True).sort_values(["date", "ticker"]).reset_index(drop=True)
prices["dollar_volume"] = prices["close"] * prices["volume"]
prices["market_cap_m"] = prices["close"] * prices["ticker"].map(meta.set_index("ticker")["shares_outstanding_m"])

# Add suspensions / missingness
susp_mask = rng.random(len(prices)) < 0.0025
prices.loc[susp_mask, ["open", "high", "low", "close", "volume", "dollar_volume"]] = np.nan

# Duplicate a small sample intentionally for de-duplication drills
dupe_rows = prices.sample(40, random_state=SEED)
prices_with_dupes = pd.concat([prices, dupe_rows], ignore_index=True).sort_values(["date", "ticker"]).reset_index(drop=True)

prices.head()


,date,ticker,open,high,low,close,volume,ret_1d,dollar_volume,market_cap_m
0,2018-01-01,STK114,32.99716,34.48006,32.88517,34.19602,"1,875,889.28604",0.03424,"64,147,956.54323","40,482.20393"
1,2018-01-02,STK114,34.05827,35.01304,33.95535,34.82747,"532,670.87712",0.01847,"18,551,580.59727","41,229.72986"
2,2018-01-03,STK114,34.77533,34.87957,34.55752,34.80562,"255,077.97769",-0.00063,"8,878,148.23604","41,203.86464"
3,2018-01-04,STK114,34.84008,35.02177,33.68647,33.84211,"2,153,070.52338",-0.02768,"72,864,443.64518","40,063.22655"
4,2018-01-04,STK129,95.37483,95.86763,94.28590,94.75851,"213,712.57938",0.00104,"20,251,085.19489","136,374.14809"


In [4]:

# --------------------------------------
# Quarterly fundamentals with announce lag
# --------------------------------------
quarter_ends = pd.date_range("2017-12-31", "2022-12-31", freq="Q")
fund_rows = []

for _, row in meta.iterrows():
    base_assets = rng.uniform(500, 25000)
    profitability = 0.02 + 0.02 * row["quality_tilt"] + rng.normal(0, 0.01)
    leverage_base = np.clip(0.8 - 0.15 * row["quality_tilt"] + rng.normal(0, 0.15), 0.1, 2.5)
    sales_base = rng.uniform(100, 12000)

    assets = base_assets
    sales = sales_base
    book = base_assets * rng.uniform(0.35, 0.80)

    for qd in quarter_ends:
        announce_lag_days = int(rng.integers(18, 55))
        announce_date = pd.bdate_range(qd, periods=announce_lag_days + 1)[-1]

        asset_growth = rng.normal(0.01 + 0.01 * row["growth_tilt"], 0.03)
        sales_growth = rng.normal(0.015 + 0.02 * row["growth_tilt"], 0.05)

        assets = max(50, assets * (1 + asset_growth))
        sales = max(20, sales * (1 + sales_growth))
        book = max(10, book * (1 + rng.normal(0.01 + 0.005 * row["value_tilt"], 0.03)))

        net_income = sales * (profitability + rng.normal(0, 0.015))
        total_debt = assets * np.clip(leverage_base + rng.normal(0, 0.08), 0.02, 3.0) / 2
        cash = assets * np.clip(rng.normal(0.10 + 0.02 * row["quality_tilt"], 0.04), 0.01, 0.40)

        fund_rows.append({
            "ticker": row["ticker"],
            "fiscal_q_end": qd,
            "announce_date": announce_date,
            "sales_m": sales,
            "assets_m": assets,
            "book_equity_m": book,
            "net_income_m": net_income,
            "debt_m": total_debt,
            "cash_m": cash,
        })

fundamentals = pd.DataFrame(fund_rows).sort_values(["ticker", "announce_date"]).reset_index(drop=True)

# Add missing fundamentals for realism
miss_mask = rng.random(len(fundamentals)) < 0.03
fundamentals.loc[miss_mask, ["sales_m", "assets_m", "book_equity_m", "net_income_m"]] = np.nan

fundamentals.head()


/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/1615225350.py:4: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarter_ends = pd.date_range("2017-12-31", "2022-12-31", freq="Q")


,ticker,fiscal_q_end,announce_date,sales_m,assets_m,book_equity_m,net_income_m,debt_m,cash_m
0,STK000,2017-12-31,2018-02-20,"4,865.30129","13,234.25362","6,315.00223",78.23690,"7,573.11155","1,010.06078"
1,STK000,2018-03-31,2018-05-04,"4,702.81445","13,174.63463","6,335.33212",194.58026,"7,026.89899",966.21989
2,STK000,2018-06-30,2018-08-23,"4,504.56659","13,579.98732","6,286.90831",124.54123,"7,178.89294","1,756.41990"
3,STK000,2018-09-30,2018-11-07,"4,525.16578","13,839.55260","6,265.24644",155.70112,"7,791.40148","1,163.23959"
4,STK000,2018-12-31,2019-02-07,"4,743.41914","13,778.45097","6,301.01634",48.80552,"7,838.72402",995.55203


In [5]:

# --------------------------------------
# Build a base panel and some quick checks
# --------------------------------------
panel = prices.merge(
    meta[["ticker", "sector", "industry", "country", "exchange", "shares_outstanding_m", "ipo_date", "delist_date"]],
    on="ticker",
    how="left"
).sort_values(["date", "ticker"]).reset_index(drop=True)

print("prices shape:", prices.shape)
print("prices_with_dupes shape:", prices_with_dupes.shape)
print("fundamentals shape:", fundamentals.shape)
print("panel shape:", panel.shape)
print("\nDate range:", panel["date"].min(), "to", panel["date"].max())
print("Unique tickers:", panel["ticker"].nunique())

panel.head()


prices shape: (173957, 10)
prices_with_dupes shape: (173997, 10)
fundamentals shape: (3780, 9)
panel shape: (173957, 17)

Date range: 2018-01-01 00:00:00 to 2022-12-30 00:00:00
Unique tickers: 180


,date,ticker,open,high,low,close,volume,ret_1d,dollar_volume,market_cap_m,sector,industry,country,exchange,shares_outstanding_m,ipo_date,delist_date
0,2018-01-01,STK114,32.99716,34.48006,32.88517,34.19602,"1,875,889.28604",0.03424,"64,147,956.54323","40,482.20393",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
1,2018-01-02,STK114,34.05827,35.01304,33.95535,34.82747,"532,670.87712",0.01847,"18,551,580.59727","41,229.72986",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
2,2018-01-03,STK114,34.77533,34.87957,34.55752,34.80562,"255,077.97769",-0.00063,"8,878,148.23604","41,203.86464",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
3,2018-01-04,STK114,34.84008,35.02177,33.68647,33.84211,"2,153,070.52338",-0.02768,"72,864,443.64518","40,063.22655",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
4,2018-01-04,STK129,95.37483,95.86763,94.28590,94.75851,"213,712.57938",0.00104,"20,251,085.19489","136,374.14809",Financials,Financials_4,CA,TSX,"1,439.17576",2018-01-04,2022-12-30


## Section 1 - Tabular / panel data wrangling steps

This section focuses on:
- deduplication
- point-in-time joins
- reshaping between long and wide formats
- groupby transforms
- universe filters
- missing data / stale data handling
- leakage-safe target creation


### Step 1.1 - Remove duplicates cleanly


Using `prices_with_dupes`, create `prices_deduped` that keeps a single row per `(date, ticker)`.

Requirements:
- keep the last occurrence
- sort by `date, ticker`
- assert there are no duplicate `(date, ticker)` keys


In [ ]:

# TODO: create prices_deduped from prices_with_dupes
prices_deduped = None

# Suggested checks:
# assert prices_deduped.duplicated(["date", "ticker"]).sum() == 0

# SOLUTION:
# prices_deduped = (
#     prices_with_dupes
#     .drop_duplicates(subset=["date", "ticker"], keep="last")
#     .sort_values(["date", "ticker"])
#     .reset_index(drop=True)
# )
# assert prices_deduped.duplicated(["date", "ticker"]).sum() == 0


In [6]:
prices_deduped = prices.drop_duplicates(subset=['date', 'ticker'], keep="last").sort_values(['date', 'ticker'])

In [7]:
assert prices_deduped.duplicated(subset=['date', 'ticker']).sum() == 0

### Step 1.2 - Filter to a valid live universe


Create `panel_live` that only includes rows where:
- `date >= ipo_date`
- `date <= delist_date`
- `close` is non-null
- `close >= 1`
- `dollar_volume >= 1e6`


In [ ]:

# TODO: create panel_live from panel
panel_live = None

# SOLUTION:
# panel_live = panel.loc[
#     (panel["date"] >= panel["ipo_date"])
#     & (panel["date"] <= panel["delist_date"])
#     & panel["close"].notna()
#     & (panel["close"] >= 1)
#     & (panel["dollar_volume"] >= 1e6)
# ].copy()


In [8]:
mask = (panel.date >= panel.ipo_date) & (panel.date <= panel.delist_date) & (panel.close.notnull()) & (panel.close >= 1) & (panel.dollar_volume >= 1e6)
panel_live = panel.loc[mask]

In [9]:
panel_live

,date,ticker,open,high,low,close,volume,ret_1d,dollar_volume,market_cap_m,sector,industry,country,exchange,shares_outstanding_m,ipo_date,delist_date
0,2018-01-01,STK114,32.99716,34.48006,32.88517,34.19602,"1,875,889.28604",0.03424,"64,147,956.54323","40,482.20393",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
1,2018-01-02,STK114,34.05827,35.01304,33.95535,34.82747,"532,670.87712",0.01847,"18,551,580.59727","41,229.72986",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
2,2018-01-03,STK114,34.77533,34.87957,34.55752,34.80562,"255,077.97769",-0.00063,"8,878,148.23604","41,203.86464",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
3,2018-01-04,STK114,34.84008,35.02177,33.68647,33.84211,"2,153,070.52338",-0.02768,"72,864,443.64518","40,063.22655",Tech,Tech_2,US,NASDAQ,"1,183.82777",2018-01-01,2022-12-30
4,2018-01-04,STK129,95.37483,95.86763,94.28590,94.75851,"213,712.57938",0.00104,"20,251,085.19489","136,374.14809",Financials,Financials_4,CA,TSX,"1,439.17576",2018-01-04,2022-12-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173952,2022-12-30,STK173,128.22615,128.39537,127.61450,128.24546,"576,695.13747",-0.00377,"73,958,534.75498","235,913.18294",Health,Health_3,US,NASDAQ,"1,839.54409",2020-03-03,2022-12-30
173953,2022-12-30,STK175,72.67528,73.30267,71.85022,72.23902,"1,035,193.64151",-0.00294,"74,781,378.18546","49,824.31511",Financials,Financials_2,US,NYSE,689.71468,2018-10-08,2022-12-30
173954,2022-12-30,STK177,45.40819,46.07167,44.85766,45.98259,"357,872.93473",0.01380,"16,455,923.06999","62,693.22484",Materials,Materials_1,US,NYSE,"1,363.41233",2019-04-19,2022-12-30
173955,2022-12-30,STK178,153.18074,153.76341,151.73642,152.76418,"819,535.36430",-0.00237,"125,195,645.63204","263,200.10877",Energy,Energy_3,US,NASDAQ,"1,722.91773",2019-10-31,2022-12-30


### Step 1.3 - Compute forward returns without leakage


For each ticker, compute:
- `ret_fwd_1d`: next-day return
- `ret_fwd_5d`: cumulative forward 5-business-day return using close-to-close daily returns

Use only information available at date `t` to predict returns after `t`.


In [ ]:

# TODO: add ret_fwd_1d and ret_fwd_5d to panel_live
# Hint: group by ticker and use shift(-1), rolling windows, or log-return accumulation carefully.

# SOLUTION:
# panel_live = panel_live.sort_values(["ticker", "date"]).copy()
# g = panel_live.groupby("ticker")
# panel_live["ret_fwd_1d"] = g["ret_1d"].shift(-1)
# panel_live["ret_fwd_5d"] = (
#     g["ret_1d"]
#     .transform(lambda s: (1 + s).shift(-1).rolling(5, min_periods=5).apply(np.prod, raw=True) - 1)
# )


In [10]:
panel_live_idx = panel_live.set_index(['date', 'ticker'])

In [11]:
panel_live_idx['ret_fwd_1d'] = panel_live_idx['ret_1d'].unstack().shift(-1).stack()

In [12]:
fwd_ret_unstack = (panel_live_idx['ret_1d'].unstack().rolling(5, min_periods=5).apply(lambda x : ((x + 1).cumprod() - 1)[-1])).shift(-5)

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/2226244051.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fwd_ret_unstack = (panel_live_idx['ret_1d'].unstack().rolling(5, min_periods=5).apply(lambda x : ((x + 1).cumprod() - 1)[-1])).shift(-5)


In [13]:
panel_live_idx['ret_fwd_5d'] = fwd_ret_unstack.stack()

### Step 1.4 - Long to wide and back


Create a wide close-price matrix:
- index = date
- columns = ticker
- values = close

Then convert it back to long form with columns:
`date, ticker, close`

Verify equality of the non-null observations after sorting.


In [ ]:

# TODO: create close_wide and close_long_roundtrip

# SOLUTION:
# close_wide = panel.pivot(index="date", columns="ticker", values="close")
# close_long_roundtrip = (
#     close_wide
#     .stack(dropna=True)
#     .rename("close")
#     .reset_index()
#     .sort_values(["date", "ticker"])
#     .reset_index(drop=True)
# )


In [14]:
close_unstack = panel[['date', 'ticker', 'close']].set_index(['date', 'ticker']).unstack().head()

In [15]:
close_unstack.stack().reset_index(drop=False)

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/3437336258.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  close_unstack.stack().reset_index(drop=False)


,date,ticker,close
0,2018-01-01,STK114,34.19602
1,2018-01-02,STK114,34.82747
2,2018-01-03,STK114,34.80562
3,2018-01-04,STK114,33.84211
4,2018-01-04,STK129,94.75851
5,2018-01-05,STK061,57.47802
6,2018-01-05,STK114,32.52581
7,2018-01-05,STK129,91.71328


### Step 1.5 - Sector/date cross-sectional z-score


For each `date` and `sector`, compute a z-scored version of `dollar_volume` called `dv_z_by_sector`.

Use a `groupby(...).transform(...)` implementation.
Handle zero standard deviation safely.


In [16]:
panel_live['dv_z_by_sector'] = panel_live.groupby(['date', 'sector'])['dollar_volume'].transform(lambda x : (x - x.mean())/x.std() if x.std() > 0 else 0.0)

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/2318464945.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_live['dv_z_by_sector'] = panel_live.groupby(['date', 'sector'])['dollar_volume'].transform(lambda x : (x - x.mean())/x.std() if x.std() > 0 else 0.0)


### Step 1.6 - Cross-sectional rank by date


Compute:
- `close_rank_pct`: percentile rank of `close` within each date
- `mcap_rank_pct`: percentile rank of `market_cap_m` within each date

Use ties handled by average rank.


In [ ]:

# TODO: create close_rank_pct and mcap_rank_pct

# SOLUTION:
# panel_live["close_rank_pct"] = panel_live.groupby("date")["close"].rank(pct=True, method="average")
# panel_live["mcap_rank_pct"] = panel_live.groupby("date")["market_cap_m"].rank(pct=True, method="average")


In [17]:
panel_live['close_rank_pct'] = panel_live.groupby('date')['close'].transform(lambda x : (x.rank() - 0.5)/len(x))

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/1866737147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_live['close_rank_pct'] = panel_live.groupby('date')['close'].transform(lambda x : (x.rank() - 0.5)/len(x))


In [18]:
panel_live['mcap_rank_pct'] = panel_live.groupby('date')['market_cap_m'].transform(lambda x : (x.rank() - 0.5)/len(x))

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/3591306226.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_live['mcap_rank_pct'] = panel_live.groupby('date')['market_cap_m'].transform(lambda x : (x.rank() - 0.5)/len(x))


### Step 1.7 - Point-in-time join fundamentals to daily panel


Create a PIT-joined daily panel where each `(date, ticker)` row has the **latest announced** fundamental data available as of that date.

Use `pd.merge_asof` or an equivalent PIT-safe method.

Expected output columns should include:
- `sales_m`
- `assets_m`
- `book_equity_m`
- `net_income_m`
- `debt_m`
- `cash_m`

Name the result `panel_pit`.


In [ ]:

# TODO: create panel_pit as a point-in-time join between panel_live and fundamentals

# SOLUTION:
# left = panel_live.sort_values(["ticker", "date"]).copy()
# right = fundamentals.sort_values(["ticker", "announce_date"]).copy()
# panel_pit = pd.merge_asof(
#     left,
#     right,
#     by="ticker",
#     left_on="date",
#     right_on="announce_date",
#     direction="backward",
# )


In [19]:
panel_live.sort_values(['date', 'ticker'], inplace=True)
fundamentals.sort_values(['announce_date', 'ticker'], inplace=True)

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/3699458048.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_live.sort_values(['date', 'ticker'], inplace=True)


In [20]:
panel_pit = pd.merge_asof(panel_live, fundamentals, by='ticker', left_on='date', right_on='announce_date', direction='backward')

### Step 1.8 - Flag stale fundamentals


On `panel_pit`, compute:
- `days_since_announce`
- `is_stale_fundamental` = 1 if more than 120 business days old, else 0


In [ ]:

# TODO: create days_since_announce and is_stale_fundamental

# SOLUTION:
# panel_pit["days_since_announce"] = (panel_pit["date"] - panel_pit["announce_date"]).dt.days
# panel_pit["is_stale_fundamental"] = (panel_pit["days_since_announce"] > 120).astype(int)


In [21]:
panel_pit['days_since_announce'] = (panel_pit['date'] - panel_pit['announce_date']).dt.days

In [22]:
# stale data
panel_pit.loc[panel_pit.days_since_announce > 120]

,date,ticker,open,high,low,close,volume,ret_1d,dollar_volume,market_cap_m,sector,industry,country,exchange,shares_outstanding_m,ipo_date,delist_date,dv_z_by_sector,close_rank_pct,mcap_rank_pct,fiscal_q_end,announce_date,sales_m,assets_m,book_equity_m,net_income_m,debt_m,cash_m,days_since_announce
1983,2018-06-11,STK115,31.69517,31.79285,31.40096,31.76701,"351,440.39682",-0.00276,"11,164,208.85816","68,379.19751",Utilities,Utilities_2,UK,NASDAQ,"2,152.52264",2018-06-11,2022-12-30,-0.61213,0.16176,0.39706,2017-12-31,2018-02-06,"2,040.04330","14,490.51123","10,772.86268",43.41179,"4,113.32893",852.98551,125.00000
4472,2018-08-27,STK039,179.47716,181.49216,173.43484,174.03549,"1,272,449.89863",-0.02477,"221,451,443.80792","95,772.13559",Tech,Tech_3,US,NASDAQ,550.30232,2018-03-26,2022-12-30,1.83237,0.97170,0.57547,2018-03-31,2018-04-27,"8,438.07743","8,137.43395","4,905.57899",94.81059,"4,524.19478",147.76903,122.00000
4525,2018-08-28,STK039,173.99678,183.12261,173.41253,182.30143,"1,353,650.83166",0.04750,"246,772,487.49246","100,320.90273",Tech,Tech_3,US,NASDAQ,550.30232,2018-03-26,2022-12-30,1.80508,0.97273,0.59091,2018-03-31,2018-04-27,"8,438.07743","8,137.43395","4,905.57899",94.81059,"4,524.19478",147.76903,123.00000
4580,2018-08-29,STK039,181.60357,189.18122,179.63881,187.81925,"610,791.35756",0.03027,"114,718,377.22872","103,357.37209",Tech,Tech_3,US,NASDAQ,550.30232,2018-03-26,2022-12-30,1.60045,0.97273,0.59091,2018-03-31,2018-04-27,"8,438.07743","8,137.43395","4,905.57899",94.81059,"4,524.19478",147.76903,124.00000
4583,2018-08-29,STK052,40.00286,40.85798,39.96911,40.79159,"1,503,758.24630",0.01714,"61,340,688.19054","97,165.92455",Tech,Tech_2,CA,LSE,"2,382.00882",2018-06-07,2022-12-30,0.08925,0.22727,0.55455,2018-03-31,2018-04-30,"3,146.80999","14,897.20791","6,694.79562",94.73028,"3,526.11027","1,964.69724",121.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171444,2022-12-13,STK092,87.67652,87.98362,87.18147,87.60370,"212,016.43908",-0.00263,"18,573,423.52118","20,088.25527",Industrials,Industrials_1,US,TSX,229.30831,2018-11-29,2022-12-30,-0.57548,0.43377,0.12252,2022-06-30,2022-07-27,"2,312.68227","29,025.24828","10,227.80073",65.91261,"12,160.64291","3,757.95483",139.00000
171490,2022-12-13,STK142,48.11633,49.19975,47.72165,49.10859,"3,144,067.64072",0.02287,"154,400,732.64325","45,923.00427",Industrials,Industrials_2,US,TSX,935.13178,2019-09-19,2022-12-30,0.79208,0.23510,0.29470,2022-06-30,2022-08-05,"6,216.56505","7,305.42068","9,369.75558",430.49719,"2,103.17685",476.54693,130.00000
171543,2022-12-14,STK031,322.43065,324.11871,313.67695,315.03390,"1,307,667.25187",-0.02123,"411,959,508.70473","132,020.96687",Health,Health_3,US,TSX,419.06909,2018-12-03,2022-12-30,0.44768,0.88411,0.55298,2022-06-30,2022-08-10,"4,474.93028","22,685.53437","22,664.70324",82.02719,"5,610.97306","4,164.20982",126.00000
171584,2022-12-14,STK079,195.75614,196.60915,185.98439,186.84517,"647,622.22651",-0.04516,"121,005,085.32771","321,765.14360",Tech,Tech_1,US,NYSE,"1,722.09505",2019-11-21,2022-12-30,-0.01708,0.75828,0.86424,2022-06-30,2022-08-02,"24,112.42542","1,531.02345",887.90026,634.35423,624.40880,180.97515,134.00000


### Step 1.9 - Build a rebalance snapshot


Create `month_end_snapshot` containing the **last available trading date of each month** for each ticker.

Include:
- date
- ticker
- sector
- close
- market_cap_m
- ret_fwd_1d
- ret_fwd_5d


In [ ]:

# TODO: create month_end_snapshot

# SOLUTION:
# panel_pit["month"] = panel_pit["date"].dt.to_period("M")
# month_end_snapshot = (
#     panel_pit.sort_values(["ticker", "date"])
#     .groupby(["ticker", "month"], as_index=False)
#     .tail(1)
#     .sort_values(["date", "ticker"])
#     .reset_index(drop=True)
# )


In [23]:
month_end_snapshot = panel_pit.loc[panel_pit.date.isin(pd.date_range(panel_pit.date.min(), panel_pit.date.max(), freq='BM'))]

/var/folders/kc/7dyzg2fs2zv9v849s3n6rd4w0000gn/T/ipykernel_91109/1656893348.py:1: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  month_end_snapshot = panel_pit.loc[panel_pit.date.isin(pd.date_range(panel_pit.date.min(), panel_pit.date.max(), freq='BM'))]


### Step 1.10 - Coverage diagnostics table


Create a daily diagnostics DataFrame with one row per `date` containing:
- `n_names`
- `n_sectors`
- `pct_missing_close`
- `pct_missing_book`
- `median_dollar_volume`


In [ ]:

# TODO: create daily_diag

# SOLUTION:
# daily_diag = (
#     panel_pit.groupby("date")
#     .agg(
#         n_names=("ticker", "nunique"),
#         n_sectors=("sector", "nunique"),
#         pct_missing_close=("close", lambda s: s.isna().mean()),
#         pct_missing_book=("book_equity_m", lambda s: s.isna().mean()),
#         median_dollar_volume=("dollar_volume", "median"),
#     )
#     .reset_index()
# )


In [24]:
panel_pit.groupby('date').agg(
    n_names=("ticker", "nunique"), 
    n_sectors=("sector", "nunique"),
    pct_missing_close=("close", lambda s: s.isna().mean()),
    pct_missing_book=("book_equity_m", lambda s: s.isna().mean()),
    median_dollar_volume=("dollar_volume", "median")
)

,n_names,n_sectors,pct_missing_close,pct_missing_book,median_dollar_volume
date,,,,,
2018-01-01,1,1,0.00000,1.00000,"64,147,956.54323"
2018-01-02,1,1,0.00000,1.00000,"18,551,580.59727"
2018-01-03,1,1,0.00000,1.00000,"8,878,148.23604"
2018-01-04,2,2,0.00000,1.00000,"46,557,764.42004"
2018-01-05,3,3,0.00000,1.00000,"29,908,644.83128"
...,...,...,...,...,...
2022-12-26,150,10,0.00000,0.02667,"79,351,545.43723"
2022-12-27,150,10,0.00000,0.02667,"63,050,878.58308"
2022-12-28,150,10,0.00000,0.02667,"68,899,741.76951"


## Section 2 - Vectorized pandas / NumPy coding steps

This section focuses on:
- avoiding loops
- broadcasting
- masked array logic
- cumulative / rolling transformations
- efficient portfolio and diagnostic computations


### Step 2.1 - Vectorized true range


Using OHLC data on `panel_live`, compute a vectorized `true_range` column defined as:

`max(high - low, abs(high - prev_close), abs(low - prev_close))`

where `prev_close` is previous day's close for the same ticker.


In [ ]:

# TODO: compute true_range vectorized without per-row Python loops

# SOLUTION:
# panel_live = panel_live.sort_values(["ticker", "date"]).copy()
# panel_live["prev_close"] = panel_live.groupby("ticker")["close"].shift(1)
# a = panel_live["high"] - panel_live["low"]
# b = (panel_live["high"] - panel_live["prev_close"]).abs()
# c = (panel_live["low"] - panel_live["prev_close"]).abs()
# panel_live["true_range"] = np.nanmax(np.vstack([a.values, b.values, c.values]), axis=0)


In [42]:
panel_live_idx = panel_live.set_index(['date', 'ticker']).sort_index()
panel_live_idx['prev_close'] = panel_live_idx['close'].unstack().shift(1).stack()
a = panel_live_idx['high'] - panel_live_idx['low']
b = abs(panel_live_idx['high'] - panel_live_idx['prev_close'])
c = abs(panel_live_idx['low'] - panel_live_idx['prev_close'])
panel_live_idx['true_range'] = np.nanmax(np.column_stack([a, b, c]), axis=1)
panel_live_idx['true_range']

date        ticker
2018-01-01  STK114   1.59489
2018-01-02  STK114   1.05769
2018-01-03  STK114   0.32205
2018-01-04  STK114   1.33530
            STK129   1.58173
                       ...  
2022-12-30  STK173   1.11587
            STK175   1.45246
            STK177   1.21401
            STK178   2.02699
            STK179   6.81247
Name: true_range, Length: 173480, dtype: float64

### Step 2.2 - Vectorized gap and intraday return


Compute:
- `gap_ret = open / prev_close - 1`
- `intraday_ret = close / open - 1`

Use vectorized arithmetic only.


In [ ]:

# TODO: compute gap_ret and intraday_ret

# SOLUTION:
# panel_live["gap_ret"] = panel_live["open"] / panel_live["prev_close"] - 1
# panel_live["intraday_ret"] = panel_live["close"] / panel_live["open"] - 1


### Step 2.3 - Winsorize by date using percentiles


For each date, winsorize `ret_1d` at the 1st and 99th percentiles across names.
Store in `ret_1d_winsor`.

Do this without explicit Python loops over dates.


In [ ]:

# TODO: winsorize ret_1d cross-sectionally by date

# SOLUTION:
# q01 = panel_live.groupby("date")["ret_1d"].transform(lambda s: s.quantile(0.01))
# q99 = panel_live.groupby("date")["ret_1d"].transform(lambda s: s.quantile(0.99))
# panel_live["ret_1d_winsor"] = panel_live["ret_1d"].clip(lower=q01, upper=q99)


### Step 2.4 - Vectorized long-short decile spread


At each date:
- assign names to deciles using `dv_z_by_sector`
- compute equal-weight mean `ret_fwd_5d` for top and bottom deciles
- create a daily `top_minus_bottom` spread series

Avoid iterating over dates manually.


In [ ]:

# TODO: create a decile spread time series named decile_spread_ts

# SOLUTION:
# panel_live["dv_decile"] = panel_live.groupby("date")["dv_z_by_sector"].transform(
#     lambda s: pd.qcut(s.rank(method="first"), 10, labels=False, duplicates="drop") + 1
# )
# decile_spread_ts = (
#     panel_live.loc[panel_live["dv_decile"].isin([1, 10])]
#     .groupby(["date", "dv_decile"])["ret_fwd_5d"]
#     .mean()
#     .unstack()
# )
# decile_spread_ts["top_minus_bottom"] = decile_spread_ts[10] - decile_spread_ts[1]


### Step 2.5 - Vectorized market-neutral weights


For a chosen cross-sectional score, create daily market-neutral weights:

1. demean the score by date
2. normalize so sum of absolute weights = 1 per date
3. store as `w_score_neutral`

You may use `dv_z_by_sector` as the score.


In [ ]:

# TODO: create w_score_neutral

# SOLUTION:
# score_demeaned = panel_live["dv_z_by_sector"] - panel_live.groupby("date")["dv_z_by_sector"].transform("mean")
# denom = score_demeaned.groupby(panel_live["date"]).transform(lambda s: s.abs().sum())
# panel_live["w_score_neutral"] = score_demeaned / denom


### Step 2.6 - Vectorized portfolio return and turnover


Using `w_score_neutral` and `ret_fwd_1d`:

- compute daily portfolio return
- compute one-way turnover assuming next day's holdings are today's target weights
- output a DataFrame `portfolio_diag` with columns:
  - `port_ret`
  - `gross`
  - `net`
  - `turnover_1way`

Hint: turnover can be approximated with changes in weights across dates after aligning by ticker.


In [ ]:

# TODO: create portfolio_diag

# SOLUTION:
# panel_live = panel_live.sort_values(["date", "ticker"]).copy()
# port_ret = panel_live.groupby("date").apply(lambda df: np.nansum(df["w_score_neutral"] * df["ret_fwd_1d"]))
# gross = panel_live.groupby("date")["w_score_neutral"].apply(lambda s: s.abs().sum())
# net = panel_live.groupby("date")["w_score_neutral"].sum()
# panel_live["w_prev"] = panel_live.groupby("ticker")["w_score_neutral"].shift(1)
# turnover = panel_live.groupby("date").apply(lambda df: 0.5 * np.nansum(np.abs(df["w_score_neutral"] - df["w_prev"].fillna(0))))
# portfolio_diag = pd.concat(
#     [port_ret.rename("port_ret"), gross.rename("gross"), net.rename("net"), turnover.rename("turnover_1way")],
#     axis=1
# ).reset_index()


### Step 2.7 - Broadcasting sector demeaning in wide format


Create a wide close-return matrix and then, for a single date, show how to subtract sector means using vectorized broadcasting.

Goal:
- compute a one-date residualized cross-section of returns after sector demeaning
- no Python loop over tickers


In [ ]:

# TODO: demonstrate broadcasting-based sector demeaning for one chosen date

example_date = panel_live["date"].dropna().iloc[200]

# SOLUTION:
# one_day = panel_live.loc[panel_live["date"] == example_date, ["ticker", "sector", "ret_1d"]].dropna()
# sector_means = one_day.groupby("sector")["ret_1d"].transform("mean")
# one_day["ret_sector_demeaned"] = one_day["ret_1d"] - sector_means
# one_day.head()


### Step 2.8 - Vectorized trailing max drawdown by ticker


For each ticker, compute:
- cumulative return index from `ret_1d`
- running peak
- drawdown
- trailing max drawdown

Store drawdown-related columns on `panel_live`.


In [ ]:

# TODO: compute cum_index, running_peak, drawdown, trailing_max_drawdown by ticker

# SOLUTION:
# panel_live = panel_live.sort_values(["ticker", "date"]).copy()
# panel_live["cum_index"] = panel_live.groupby("ticker")["ret_1d"].transform(lambda s: (1 + s).cumprod())
# panel_live["running_peak"] = panel_live.groupby("ticker")["cum_index"].transform(lambda s: s.cummax())
# panel_live["drawdown"] = panel_live["cum_index"] / panel_live["running_peak"] - 1
# panel_live["trailing_max_drawdown"] = panel_live.groupby("ticker")["drawdown"].transform(lambda s: s.cummin())


## Section 3 - Time-series and cross-sectional feature engineering steps

This section focuses on:
- leakage-safe rolling features
- momentum / reversal
- volatility / liquidity
- PIT-safe valuation and quality features
- cross-sectional normalization
- neutralization / residualization


### Step 3.1 - Basic lagged returns features


For each ticker, compute:
- `ret_lag_1`
- `ret_lag_5`
- `mom_5d`: cumulative return over t-5 to t-1
- `mom_21d`: cumulative return over t-21 to t-1
- `rev_5d`: negative of prior 5-day return


In [ ]:

# TODO: compute lagged-return and momentum/reversal features

# SOLUTION:
# g = panel_pit.sort_values(["ticker", "date"]).groupby("ticker")
# panel_pit["ret_lag_1"] = g["ret_1d"].shift(1)
# panel_pit["ret_lag_5"] = g["ret_1d"].shift(5)
# panel_pit["mom_5d"] = g["ret_1d"].transform(lambda s: (1 + s).shift(1).rolling(5, min_periods=5).apply(np.prod, raw=True) - 1)
# panel_pit["mom_21d"] = g["ret_1d"].transform(lambda s: (1 + s).shift(1).rolling(21, min_periods=21).apply(np.prod, raw=True) - 1)
# panel_pit["rev_5d"] = -panel_pit["mom_5d"]


### Step 3.2 - Rolling volatility and downside volatility


For each ticker, compute:
- `vol_21d`: rolling 21-day std of daily returns using only data up to t-1
- `downside_vol_21d`: same but only on negative returns

Be careful about leakage.


In [ ]:

# TODO: compute vol_21d and downside_vol_21d

# SOLUTION:
# g = panel_pit.sort_values(["ticker", "date"]).groupby("ticker")
# panel_pit["vol_21d"] = g["ret_1d"].transform(lambda s: s.shift(1).rolling(21, min_periods=21).std())
# panel_pit["downside_vol_21d"] = g["ret_1d"].transform(
#     lambda s: s.where(s < 0).shift(1).rolling(21, min_periods=10).std()
# )


### Step 3.3 - Rolling liquidity features


Compute:
- `adv_21d`: 21-day average dollar volume using only prior days
- `log_adv_21d`
- `turnover_21d`: average volume / shares outstanding over trailing 21 days


In [ ]:

# TODO: compute adv_21d, log_adv_21d, turnover_21d

# SOLUTION:
# g = panel_pit.sort_values(["ticker", "date"]).groupby("ticker")
# panel_pit["adv_21d"] = g["dollar_volume"].transform(lambda s: s.shift(1).rolling(21, min_periods=21).mean())
# panel_pit["log_adv_21d"] = np.log(panel_pit["adv_21d"])
# panel_pit["turnover_21d"] = g["volume"].transform(lambda s: s.shift(1).rolling(21, min_periods=21).mean()) / (
#     panel_pit["shares_outstanding_m"] * 1_000_000
# )


### Step 3.4 - PIT-safe valuation and quality ratios


Using the PIT-joined fundamentals, create:
- `book_to_mkt`
- `earnings_to_mkt`
- `debt_to_assets`
- `cash_to_assets`
- `roa`

Use current daily market cap with latest announced fundamentals.


In [ ]:

# TODO: compute valuation and quality ratios

# SOLUTION:
# panel_pit["book_to_mkt"] = panel_pit["book_equity_m"] / panel_pit["market_cap_m"]
# panel_pit["earnings_to_mkt"] = panel_pit["net_income_m"] / panel_pit["market_cap_m"]
# panel_pit["debt_to_assets"] = panel_pit["debt_m"] / panel_pit["assets_m"]
# panel_pit["cash_to_assets"] = panel_pit["cash_m"] / panel_pit["assets_m"]
# panel_pit["roa"] = panel_pit["net_income_m"] / panel_pit["assets_m"]


### Step 3.5 - Growth features from fundamentals


Using quarterly PIT fundamentals, compute:
- `sales_growth_yoy`
- `asset_growth_yoy`

These should compare the latest announced quarter to the quarter announced roughly 4 quarters earlier for the same ticker.


In [ ]:

# TODO: compute yoy growth features from fundamentals first, then join if needed

# SOLUTION:
# fundamentals = fundamentals.sort_values(["ticker", "announce_date"]).copy()
# fundamentals["sales_growth_yoy"] = fundamentals.groupby("ticker")["sales_m"].pct_change(4)
# fundamentals["asset_growth_yoy"] = fundamentals.groupby("ticker")["assets_m"].pct_change(4)
#
# right = fundamentals.sort_values(["ticker", "announce_date"])
# left = panel_pit.drop(columns=[c for c in ["sales_growth_yoy", "asset_growth_yoy"] if c in panel_pit.columns]).sort_values(["ticker", "date"])
# panel_pit = pd.merge_asof(
#     left,
#     right[["ticker", "announce_date", "sales_growth_yoy", "asset_growth_yoy"]].sort_values(["ticker", "announce_date"]),
#     by="ticker",
#     left_on="date",
#     right_on="announce_date",
#     direction="backward",
# )


### Step 3.6 - Cross-sectional standardization by date


For the following raw features, create same-date z-scores:
- `mom_21d`
- `vol_21d`
- `book_to_mkt`
- `roa`
- `log_adv_21d`

Use suffix `_z`.


In [ ]:

# TODO: create z-scored versions of selected features

# SOLUTION:
# feature_cols = ["mom_21d", "vol_21d", "book_to_mkt", "roa", "log_adv_21d"]
# for col in feature_cols:
#     panel_pit[f"{col}_z"] = panel_pit.groupby("date")[col].transform(
#         lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0
#     )


### Step 3.7 - Sector-neutral z-scores


Create sector-neutral versions of:
- `mom_21d`
- `book_to_mkt`
- `roa`

For each date and sector:
1. demean
2. divide by within-sector std


In [ ]:

# TODO: create sector-neutral z-scores

# SOLUTION:
# feature_cols = ["mom_21d", "book_to_mkt", "roa"]
# for col in feature_cols:
#     panel_pit[f"{col}_sector_z"] = panel_pit.groupby(["date", "sector"])[col].transform(
#         lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0
#     )


### Step 3.8 - Composite alpha score


Create a composite score using:
- positive momentum (`mom_21d_sector_z`)
- value (`book_to_mkt_sector_z`)
- quality (`roa_sector_z`)
- negative volatility (`-vol_21d_z`)

Requirements:
- simple equal-weight average across available sub-signals
- store as `alpha_composite`


In [ ]:

# TODO: create alpha_composite

# SOLUTION:
# comp_cols = ["mom_21d_sector_z", "book_to_mkt_sector_z", "roa_sector_z"]
# # If book_to_mkt_sector_z / roa_sector_z do not yet exist due to naming, create them first or adapt names.
# panel_pit["neg_vol_z"] = -panel_pit["vol_21d_z"]
# comp_cols = [c for c in ["mom_21d_sector_z", "book_to_mkt_sector_z", "roa_sector_z", "neg_vol_z"] if c in panel_pit.columns]
# panel_pit["alpha_composite"] = panel_pit[comp_cols].mean(axis=1)


### Step 3.9 - Residualize a feature versus size


For each date, residualize `book_to_mkt_z` against `log_adv_21d_z` and `mcap_rank_pct`.

Store residuals as `book_to_mkt_resid`.

You can implement with:
- a per-date regression using `np.linalg.lstsq`
- or another vectorized / grouped approach

This is intentionally harder and closer to interview difficulty.


In [ ]:

# TODO: residualize book_to_mkt_z against size/liquidity controls by date
# One straightforward interview-safe approach is a groupby-apply returning residuals.

# SOLUTION:
# def residualize_one_day(df, y_col, x_cols):
#     tmp = df[[y_col] + x_cols].dropna().copy()
#     out = pd.Series(np.nan, index=df.index)
#     if len(tmp) < len(x_cols) + 2:
#         return out
#     X = tmp[x_cols].to_numpy()
#     X = np.column_stack([np.ones(len(X)), X])
#     y = tmp[y_col].to_numpy()
#     beta, *_ = np.linalg.lstsq(X, y, rcond=None)
#     resid = y - X @ beta
#     out.loc[tmp.index] = resid
#     return out
#
# panel_pit["mcap_rank_pct"] = panel_pit.groupby("date")["market_cap_m"].rank(pct=True)
# panel_pit["book_to_mkt_resid"] = (
#     panel_pit.groupby("date", group_keys=False)
#     .apply(lambda df: residualize_one_day(df, "book_to_mkt_z", ["log_adv_21d_z", "mcap_rank_pct"]))
# )


### Step 3.10 - Information coefficient prep


Prepare a daily IC dataset using `alpha_composite` and `ret_fwd_5d`:
- keep rows where both are non-null
- compute same-date Pearson IC
- compute same-date Spearman/rank IC
- output `ic_ts`


In [ ]:

# TODO: create ic_ts

# SOLUTION:
# def one_day_ic(df):
#     x = df["alpha_composite"]
#     y = df["ret_fwd_5d"]
#     return pd.Series({
#         "ic_pearson": x.corr(y, method="pearson"),
#         "ic_spearman": x.corr(y, method="spearman"),
#         "n": len(df)
#     })
#
# ic_ts = (
#     panel_pit[["date", "alpha_composite", "ret_fwd_5d"]]
#     .dropna()
#     .groupby("date")
#     .apply(one_day_ic)
#     .reset_index()
# )
